# SUB003 -- Template + Neural Refinement Pipeline (No Internet)

**Competition**: Stanford RNA 3D Folding Part 2  
**Metric**: TM-score (higher is better)  
**Lineage**: IT001 -> IT002 -> SUB001 (inf) -> SUB002 (7.5e33) -> SUB003

Fixes from SUB002:
- **No internet**: Loads pre-built template DB from dataset (jerryishihara/rna-pdb-template-db)
- **Zero-init head**: Model starts by predicting zero delta (= pure template), preventing initial loss explosion
- **Huber loss**: Robust to outlier residues where template is far from ground truth
- **Scaled delta output**: `tanh * scale_factor` constrains predictions
- **Cached template predictions**: Template search runs once per target

In [ ]:
import json
import pickle
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

BASE = "/kaggle/input/stanford-rna-3d-folding-2"
WORK = "/kaggle/working"
DB_DIR = "/kaggle/input/rna-pdb-template-db"

CFG = {
    "top_k_templates": 5,
    "min_identity": 0.2,
    "refine_hidden": 128,
    "refine_layers": 4,
    "refine_kernel": 5,
    "refine_dropout": 0.15,
    "refine_lr": 1e-4,
    "refine_epochs": 40,
    "refine_batch": 4,
    "noise_std": 0.5,
    "delta_scale": 2.0,
}

In [ ]:
def safe_read_csv(path, label=""):
    try:
        df = pd.read_csv(path)
        print(f"Loaded {label}: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"WARNING: {label} not found at {path}")
        return pd.DataFrame()

test_sequences = safe_read_csv(f"{BASE}/test_sequences.csv", "test_sequences")
sample_sub = safe_read_csv(f"{BASE}/sample_submission.csv", "sample_submission")
val_sequences = safe_read_csv(f"{BASE}/validation_sequences.csv", "validation_sequences")
val_labels = safe_read_csv(f"{BASE}/validation_labels.csv", "validation_labels")

print("\n--- test_sequences columns:", list(test_sequences.columns))
print("--- sample_submission columns:", list(sample_sub.columns[:6]), "...")
print("--- val_sequences columns:", list(val_sequences.columns))
print("--- val_labels columns:", list(val_labels.columns[:6]), "...")

In [ ]:
THREE_TO_ONE = {
    "A": "A", "ADE": "A", "RA": "A",
    "C": "C", "CYT": "C", "RC": "C",
    "G": "G", "GUA": "G", "RG": "G",
    "U": "U", "URA": "U", "RU": "U",
    "DA": "A", "DC": "C", "DG": "G", "DT": "U",
}

RNA_RESIDUES = set(THREE_TO_ONE.keys())


class PDBRNADatabase:
    """Loads a pre-built RNA template database from JSON files (no internet)."""

    def __init__(self, db_dir: str):
        self.db_dir = Path(db_dir)
        self.index: Dict[str, dict] = {}

    def load_database(self):
        """Load all JSON chain files from db_dir into the index."""
        json_files = sorted(self.db_dir.glob("*.json"))
        print(f"Loading {len(json_files)} template entries from {self.db_dir}...")
        for jf in tqdm(json_files, desc="Loading templates"):
            try:
                with open(jf) as f:
                    entry = json.load(f)
            except (json.JSONDecodeError, IOError):
                continue
            pdb_id = entry.get("pdb_id", jf.stem.upper())
            for chain in entry.get("chains", []):
                seq = chain.get("sequence", "")
                coords = chain.get("coords", [])
                if len(seq) < 5 or len(coords) != len(seq):
                    continue
                key = f"{pdb_id}_{chain['chain_id']}"
                self.index[key] = {
                    "pdb_id": pdb_id,
                    "chain_id": chain["chain_id"],
                    "sequence": seq,
                    "length": len(seq),
                    "coords": coords,
                }
        print(f"Database loaded: {len(self.index)} chains.")
        return len(self.index)

    def search_templates(self, query_sequence, top_k=5, min_identity=0.2):
        query_upper = query_sequence.upper()
        query_kmers = _kmer_set(query_upper, k=4)
        candidates = []
        for key, entry in self.index.items():
            tpl_kmers = _kmer_set(entry["sequence"], k=4)
            if not query_kmers or not tpl_kmers:
                continue
            jaccard = len(query_kmers & tpl_kmers) / len(query_kmers | tpl_kmers)
            if jaccard >= min_identity * 0.3:
                candidates.append((key, entry, jaccard))
        candidates.sort(key=lambda x: -x[2])
        candidates = candidates[: top_k * 5]
        results = []
        for key, entry, _ in candidates:
            identity = sequence_identity(query_upper, entry["sequence"])
            if identity >= min_identity:
                results.append({
                    "key": key,
                    "pdb_id": entry["pdb_id"],
                    "chain_id": entry["chain_id"],
                    "template_sequence": entry["sequence"],
                    "template_length": entry["length"],
                    "identity": identity,
                    "coords": entry["coords"],
                })
        results.sort(key=lambda x: -x["identity"])
        return results[:top_k]


def _kmer_set(seq, k=4):
    return {seq[i : i + k] for i in range(len(seq) - k + 1)} if len(seq) >= k else set()


def sequence_identity(seq_a, seq_b):
    alignment = needleman_wunsch(seq_a, seq_b)
    if alignment["aligned_length"] == 0:
        return 0.0
    return alignment["matches"] / alignment["aligned_length"]


def needleman_wunsch(seq_a, seq_b, match_score=2, mismatch_score=-1, gap_penalty=-2):
    n, m = len(seq_a), len(seq_b)
    dp = np.zeros((n + 1, m + 1), dtype=np.int32)
    dp[0, :] = np.arange(m + 1) * gap_penalty
    dp[:, 0] = np.arange(n + 1) * gap_penalty
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match_score if seq_a[i - 1] == seq_b[j - 1] else mismatch_score
            dp[i, j] = max(
                dp[i - 1, j - 1] + s,
                dp[i - 1, j] + gap_penalty,
                dp[i, j - 1] + gap_penalty,
            )
    aligned_a, aligned_b = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = match_score if seq_a[i - 1] == seq_b[j - 1] else mismatch_score
            if dp[i, j] == dp[i - 1, j - 1] + s:
                aligned_a.append(seq_a[i - 1])
                aligned_b.append(seq_b[j - 1])
                i -= 1
                j -= 1
                continue
        if i > 0 and dp[i, j] == dp[i - 1, j] + gap_penalty:
            aligned_a.append(seq_a[i - 1])
            aligned_b.append("-")
            i -= 1
        else:
            aligned_a.append("-")
            aligned_b.append(seq_b[j - 1])
            j -= 1
    aligned_a.reverse()
    aligned_b.reverse()
    matches = sum(a == b and a != "-" for a, b in zip(aligned_a, aligned_b))
    aligned_length = sum(a != "-" or b != "-" for a, b in zip(aligned_a, aligned_b))
    a_pos, b_pos = -1, -1
    map_a_to_b = {}
    for a_char, b_char in zip(aligned_a, aligned_b):
        if a_char != "-":
            a_pos += 1
        if b_char != "-":
            b_pos += 1
        if a_char != "-" and b_char != "-":
            map_a_to_b[a_pos] = b_pos
    return {
        "aligned_a": "".join(aligned_a),
        "aligned_b": "".join(aligned_b),
        "matches": matches,
        "aligned_length": aligned_length,
        "alignment_map_a_to_b": map_a_to_b,
    }


print("Template DB module loaded.")

In [ ]:
def generate_helix_coords(length: int) -> np.ndarray:
    rise_per_residue = 2.81
    radius = 9.4
    twist_per_residue = np.radians(32.7)
    coords = np.zeros((length, 3), dtype=np.float64)
    for i in range(length):
        angle = i * twist_per_residue
        coords[i, 0] = radius * np.cos(angle)
        coords[i, 1] = radius * np.sin(angle)
        coords[i, 2] = i * rise_per_residue
    return coords


def transfer_coordinates(query_sequence, template_sequence, template_coords, alignment_map):
    query_len = len(query_sequence)
    result = np.full((query_len, 3), np.nan, dtype=np.float64)
    for q_pos, t_pos in alignment_map.items():
        if q_pos < query_len and t_pos < len(template_coords):
            result[q_pos] = template_coords[t_pos]
    mapped_positions = sorted(alignment_map.keys())
    if not mapped_positions:
        return generate_helix_coords(query_len)
    for i in range(query_len):
        if not np.isnan(result[i, 0]):
            continue
        left = max((p for p in mapped_positions if p < i), default=None)
        right = min((p for p in mapped_positions if p > i), default=None)
        if left is not None and right is not None:
            frac = (i - left) / (right - left)
            result[i] = result[left] * (1 - frac) + result[right] * frac
        elif left is not None:
            direction = _estimate_direction(result, left, mapped_positions)
            result[i] = result[left] + direction * (i - left)
        elif right is not None:
            direction = _estimate_direction(result, right, mapped_positions)
            result[i] = result[right] - direction * (right - i)
        else:
            result[i] = [0.0, 0.0, 0.0]
    return result


def _estimate_direction(coords, anchor, mapped):
    neighbors = [p for p in mapped if p != anchor and not np.isnan(coords[p, 0])]
    if not neighbors:
        return np.array([0.0, 0.0, 2.81])
    closest = min(neighbors, key=lambda p: abs(p - anchor))
    diff = coords[anchor] - coords[closest]
    dist = np.linalg.norm(diff)
    if dist < 1e-6:
        return np.array([0.0, 0.0, 2.81])
    return diff / max(abs(anchor - closest), 1)


def kabsch_rmsd(P, Q):
    assert P.shape == Q.shape
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    p = P - centroid_P
    q = Q - centroid_Q
    H = q.T @ p
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign_matrix = np.diag([1.0, 1.0, np.sign(d)])
    R = Vt.T @ sign_matrix @ U.T
    diff = p - q @ R
    rmsd_val = float(np.sqrt((diff**2).sum() / len(P)))
    return rmsd_val, R, centroid_P - centroid_Q @ R


class TemplateModel:
    def __init__(self, database, top_k=5, min_identity=0.2):
        self.database = database
        self.top_k = top_k
        self.min_identity = min_identity

    def predict(self, sequence):
        templates = self.database.search_templates(
            sequence, top_k=self.top_k, min_identity=self.min_identity
        )
        if not templates:
            return {
                "coords": generate_helix_coords(len(sequence)),
                "method": "helix_fallback",
                "templates_used": [],
                "confidence": 0.1,
            }
        predictions = []
        weights = []
        for tpl in templates:
            tpl_coords = np.array(tpl["coords"], dtype=np.float64)
            alignment = needleman_wunsch(sequence.upper(), tpl["template_sequence"])
            transferred = transfer_coordinates(
                sequence.upper(),
                tpl["template_sequence"],
                tpl_coords,
                alignment["alignment_map_a_to_b"],
            )
            predictions.append(transferred)
            weights.append(tpl["identity"])
        w = np.array(weights, dtype=np.float64)
        w = w / w.sum()
        coords = np.zeros_like(predictions[0])
        for pred, wi in zip(predictions, w):
            coords += pred * wi
        return {
            "coords": coords,
            "method": "template",
            "templates_used": [
                {"pdb_id": t["pdb_id"], "chain_id": t["chain_id"], "identity": t["identity"]}
                for t in templates
            ],
            "confidence": float(max(weights)),
        }


print("Template model module loaded.")

In [ ]:
db = PDBRNADatabase(db_dir=DB_DIR)
num_chains = db.load_database()
print(f"\nTemplate database ready: {num_chains} chains indexed.")

In [ ]:
NT_MAP = {"A": 0, "C": 1, "G": 2, "U": 3, "T": 3}

template_model = TemplateModel(
    db, top_k=CFG["top_k_templates"], min_identity=CFG["min_identity"]
)


def build_training_data(val_seqs_df, val_labels_df):
    """Build paired (template_prediction, ground_truth) tensors from validation data.

    Returns list of dicts, one per target, with keys:
      seq_str, seq_onehot, template_coords, gt_coords_list (up to 5), confidence
    """
    coord_cols_per_struct = [[f"{ax}_{i}" for ax in ("x", "y", "z")] for i in range(1, 6)]

    targets = val_seqs_df["target_id"].unique()
    samples = []
    skipped = 0

    for tid in tqdm(targets, desc="Val predictions"):
        seq_row = val_seqs_df[val_seqs_df["target_id"] == tid]
        if seq_row.empty:
            skipped += 1
            continue
        sequence = seq_row.iloc[0]["sequence"]
        if not isinstance(sequence, str) or len(sequence) < 5:
            skipped += 1
            continue

        label_rows = val_labels_df[val_labels_df["ID"].str.startswith(tid + "_")]
        if label_rows.empty:
            skipped += 1
            continue

        pred = template_model.predict(sequence)
        tpl_coords = pred["coords"]
        confidence = pred["confidence"]

        n_res = len(sequence)
        gt_structs = []
        for cols in coord_cols_per_struct:
            if not all(c in label_rows.columns for c in cols):
                continue
            gt = np.full((n_res, 3), np.nan, dtype=np.float64)
            for _, row in label_rows.iterrows():
                parts = row["ID"].split("_")
                resid = int(parts[-1])
                idx = resid - 1
                if 0 <= idx < n_res:
                    vals = row[cols].values.astype(np.float64)
                    if not np.any(np.isnan(vals)):
                        gt[idx] = vals
            if np.isnan(gt).sum() < gt.size * 0.5:
                gt = np.nan_to_num(gt, nan=0.0)
                gt_structs.append(gt)

        if not gt_structs:
            skipped += 1
            continue

        onehot = np.zeros((n_res, 4), dtype=np.float32)
        for i, c in enumerate(sequence.upper()):
            idx = NT_MAP.get(c, -1)
            if idx >= 0:
                onehot[i, idx] = 1.0

        samples.append({
            "target_id": tid,
            "sequence": sequence,
            "seq_onehot": onehot,
            "template_coords": np.array(tpl_coords, dtype=np.float32),
            "gt_coords_list": [g.astype(np.float32) for g in gt_structs],
            "confidence": confidence,
        })

    print(f"Built {len(samples)} training samples, skipped {skipped}.")
    return samples


if not val_sequences.empty and not val_labels.empty:
    train_samples = build_training_data(val_sequences, val_labels)
else:
    train_samples = []
    print("No validation data available; will skip refinement training.")

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size, dropout, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.norm1 = nn.InstanceNorm1d(channels, affine=True)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.norm2 = nn.InstanceNorm1d(channels, affine=True)
        self.drop = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        h = self.act(self.norm1(self.conv1(x)))
        h = self.drop(h)
        h = self.norm2(self.conv2(h))
        return self.act(x + h)


class RefinementNet(nn.Module):
    """1D ResNet predicting coordinate deltas with constrained output.

    Input: normalized template_xyz (3) + onehot (4) + rel_pos (1) + confidence (1) = 9
    Output: delta corrections scaled by tanh * delta_scale
    """

    IN_DIM = 9

    def __init__(self, hidden=128, num_blocks=4, kernel_size=5, dropout=0.15, delta_scale=2.0):
        super().__init__()
        self.delta_scale = delta_scale
        self.proj = nn.Sequential(
            nn.Conv1d(self.IN_DIM, hidden, 1),
            nn.InstanceNorm1d(hidden, affine=True),
            nn.GELU(),
        )
        blocks = []
        for i in range(num_blocks):
            dilation = 2 ** (i % 3)
            blocks.append(ResidualBlock(hidden, kernel_size, dropout, dilation))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Conv1d(hidden, 3, 1)
        self._init_weights()

    def _init_weights(self):
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)
        for m in self.modules():
            if isinstance(m, nn.Conv1d) and m is not self.head:
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.proj(x)
        h = self.blocks(h)
        raw = self.head(h)
        return torch.tanh(raw) * self.delta_scale


def normalize_coords(coords):
    """Center and scale coordinates. Returns (normed, centroid, scale)."""
    centroid = coords.mean(axis=0)
    centered = coords - centroid
    scale = max(np.std(centered), 1.0)
    return (centered / scale).astype(np.float32), centroid.astype(np.float32), float(scale)


def denormalize_coords(normed, centroid, scale):
    """Reverse normalize_coords."""
    return normed * scale + centroid


def build_features(template_coords_normed, seq_onehot, confidence):
    """Build (IN_DIM, L) feature tensor from NORMALIZED coords."""
    L = len(seq_onehot)
    rel_pos = np.linspace(0.0, 1.0, L, dtype=np.float32).reshape(-1, 1)
    conf = np.full((L, 1), confidence, dtype=np.float32)
    feats = np.concatenate([
        template_coords_normed.astype(np.float32),
        seq_onehot,
        rel_pos,
        conf,
    ], axis=1)
    return feats.T


model = RefinementNet(CFG['refine_hidden'], CFG['refine_layers'],
                      CFG['refine_kernel'], CFG['refine_dropout'], CFG['delta_scale'])
print(f"RefinementNet parameters: {sum(p.numel() for p in model.parameters()):,}")
del model

In [ ]:
def train_refinement(samples, cfg):
    mk = lambda: RefinementNet(cfg["refine_hidden"], cfg["refine_layers"],
                                cfg["refine_kernel"], cfg["refine_dropout"],
                                cfg["delta_scale"]).to(DEVICE)
    if not samples:
        print("No training samples; returning untrained model.")
        return mk(), False

    model = mk()
    optimizer = optim.Adam(model.parameters(), lr=cfg["refine_lr"], weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["refine_epochs"])
    huber = nn.SmoothL1Loss()

    model.train()
    best_loss = float("inf")
    best_state = None

    for epoch in range(1, cfg["refine_epochs"] + 1):
        np.random.shuffle(samples)
        epoch_loss = 0.0
        n_ok = 0

        for sample in samples:
            tpl_c = sample["template_coords"]
            gt_list = sample["gt_coords_list"]
            gt = gt_list[np.random.randint(len(gt_list))]

            tpl_normed, centroid, scale = normalize_coords(tpl_c)
            gt_normed = ((gt - centroid) / scale).astype(np.float32)

            feats = build_features(tpl_normed, sample["seq_onehot"], sample["confidence"])
            feats_t = torch.from_numpy(feats).unsqueeze(0).to(DEVICE)
            tpl_t = torch.from_numpy(tpl_normed.T).unsqueeze(0).to(DEVICE)
            gt_t = torch.from_numpy(gt_normed.T).unsqueeze(0).to(DEVICE)

            delta = model(feats_t)
            pred = tpl_t + delta
            loss = huber(pred, gt_t)

            if not torch.isfinite(loss):
                continue

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            epoch_loss += loss.item()
            n_ok += 1

        scheduler.step()
        avg_loss = epoch_loss / max(n_ok, 1)

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{cfg['refine_epochs']}  loss={avg_loss:.6f}  "
                  f"best={best_loss:.6f}  lr={scheduler.get_last_lr()[0]:.2e}  "
                  f"ok={n_ok}/{len(samples)}")

    training_succeeded = best_loss < 10.0 and np.isfinite(best_loss)
    if best_state is not None and training_succeeded:
        model.load_state_dict(best_state)
    model.eval()
    print(f"Training complete. Best loss: {best_loss:.6f}  converged: {training_succeeded}")
    if not training_succeeded:
        print("WARNING: Refinement did not converge. Using pure template predictions.")
    return model, training_succeeded


refine_model, refine_ok = train_refinement(train_samples, CFG)
print(f"Refinement model usable: {refine_ok}")

In [ ]:
def predict_refined_from_cache(tpl_coords, confidence, sequence, onehot,
                               add_noise=0.0, use_dropout=False):
    """Refine cached template prediction. Returns (L, 3) numpy array."""
    coords = tpl_coords.copy()
    if add_noise > 0:
        coords = coords + np.random.randn(*coords.shape).astype(np.float32) * add_noise

    if not refine_ok:
        return coords

    coords_normed, centroid, scale = normalize_coords(coords)
    feats = build_features(coords_normed, onehot, confidence)
    feats_t = torch.from_numpy(feats).unsqueeze(0).to(DEVICE)
    tpl_t = torch.from_numpy(coords_normed.T).unsqueeze(0).to(DEVICE)

    if use_dropout:
        refine_model.train()
    else:
        refine_model.eval()

    with torch.no_grad():
        delta = refine_model(feats_t)

    refined_normed = (tpl_t + delta).squeeze(0).T.cpu().numpy()
    refine_model.eval()
    return denormalize_coords(refined_normed, centroid, scale)


def generate_5_structures(sequence, tpl_coords, confidence, onehot):
    """Generate 5 diverse structures from a cached template prediction."""
    ns = CFG["noise_std"]
    configs = [
        (0.0,       False),
        (ns,        True),
        (ns,        True),
        (ns * 0.5,  False),
        (ns * 1.5,  True),
    ]
    return [
        predict_refined_from_cache(tpl_coords, confidence, sequence, onehot,
                                   add_noise=n, use_dropout=d)
        for n, d in configs
    ]


if not test_sequences.empty:
    test_targets = test_sequences["target_id"].unique()
    print(f"Generating predictions for {len(test_targets)} test targets...")

    all_predictions = {}
    for tid in tqdm(test_targets, desc="Test predictions"):
        seq_row = test_sequences[test_sequences["target_id"] == tid]
        sequence = seq_row.iloc[0]["sequence"]
        if not isinstance(sequence, str) or len(sequence) == 0:
            sequence = "A"

        pred = template_model.predict(sequence)
        tpl_coords = np.array(pred["coords"], dtype=np.float32)
        confidence = pred["confidence"]

        L = len(sequence)
        onehot = np.zeros((L, 4), dtype=np.float32)
        for i, c in enumerate(sequence.upper()):
            idx = NT_MAP.get(c, -1)
            if idx >= 0:
                onehot[i, idx] = 1.0

        structs = generate_5_structures(sequence, tpl_coords, confidence, onehot)
        all_predictions[tid] = structs
    print(f"Generated predictions for {len(all_predictions)} targets.")
else:
    all_predictions = {}
    print("No test sequences found.")

In [ ]:
if not sample_sub.empty and all_predictions:
    coord_col_names = [f"{ax}_{i}" for i in range(1, 6) for ax in ("x", "y", "z")]
    existing_cols = [c for c in sample_sub.columns if c in coord_col_names]
    print(f"Submission shape: {sample_sub.shape}")
    print(f"Coordinate columns found: {len(existing_cols)}")

    for idx, row in tqdm(sample_sub.iterrows(), total=len(sample_sub), desc="Filling submission"):
        parts = row["ID"].rsplit("_", 1)
        if len(parts) != 2:
            continue
        target_id, resid_str = parts
        try:
            resid = int(resid_str)
        except ValueError:
            continue

        if target_id not in all_predictions:
            pred = template_model.predict("A")
            helix = generate_helix_coords(1)
            vals = []
            for si in range(5):
                vals.extend(helix[0].tolist())
            for ci, cn in enumerate(coord_col_names):
                if cn in sample_sub.columns:
                    sample_sub.at[idx, cn] = vals[ci]
            continue

        structs = all_predictions[target_id]
        res_idx = resid - 1

        vals = []
        for si in range(5):
            s = structs[si]
            if res_idx < len(s):
                vals.extend(s[res_idx].tolist())
            else:
                fallback = generate_helix_coords(res_idx + 1)
                vals.extend(fallback[res_idx].tolist())

        for ci, cn in enumerate(coord_col_names):
            if cn in sample_sub.columns:
                sample_sub.at[idx, cn] = vals[ci]

    sample_sub = sample_sub.fillna(0.0)
    output_path = f"{WORK}/submission.csv"
    sample_sub.to_csv(output_path, index=False)
    print(f"\nSubmission saved to {output_path}")
    print(f"Shape: {sample_sub.shape}")
    print(sample_sub.head())
else:
    print("Cannot generate submission: missing test data or predictions.")
    if sample_sub.empty:
        print("sample_submission.csv was not loaded.")
    if not all_predictions:
        print("No predictions were generated.")

In [ ]:
print("=" * 60)
print("SUB003 Pipeline Complete")
print("=" * 60)
print(f"  Device used      : {DEVICE}")
print(f"  Template DB size : {len(db.index)} chains")
print(f"  Training samples : {len(train_samples)}")
print(f"  Refinement OK    : {refine_ok}")
print(f"  Test targets     : {len(all_predictions)}")
print(f"  Output           : {WORK}/submission.csv")
if not sample_sub.empty:
    print(f"  Submission rows  : {len(sample_sub)}")
    nan_count = sample_sub.isna().sum().sum()
    print(f"  NaN values       : {nan_count}")
print("=" * 60)